# Travail de groupe ETL

In [1]:
# Imports des bibliothèques

import glob

import pandas as pd

# afficher toutes les colonnes
pd.set_option("display.max_columns", 50)

all_files = sorted(glob.glob("*.csv"))

df_dic = {}
for file in all_files:
    nom = (
        file.replace("datasets/Olist/", "")
        .replace("olist_", "")
        .replace("_dataset", "")
        .replace(".csv", "")
    )
    df_dic[nom] = pd.read_csv(file)


def resume(df):
    """Renvoie un tableau de synthèse: une ligne par colonne  de df."""
    return pd.DataFrame(
        {
            "type": df.dtypes,
            "nb_uniques": df.nunique(),
            "maximum": df.max(),
            "minimum": df.min(),
            "nb manquants": df.isnull().sum(),
            "pct_manquants": (df.isnull().mean() * 100).round(2),
        }
    )

## 1 - Table Orders

In [2]:
print(df_dic["orders"].describe().round(2))
print("\n")
print(resume(df_dic["orders"]))

                                order_id                       customer_id  \
count                              99441                             99441   
unique                             99441                             99441   
top     e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
freq                                   1                                 1   

       order_status order_purchase_timestamp    order_approved_at  \
count         99441                    99441                99281   
unique            8                    98875                90733   
top       delivered      2018-03-31 15:08:21  2018-02-27 04:31:10   
freq          96478                        3                    9   

       order_delivered_carrier_date order_delivered_customer_date  \
count                         97658                         96476   
unique                        81018                         95664   
top             2018-05-09 15:48:00           2018-05-14

### Recherche des ecart moyens entres les colonnes

In [3]:
orders = df_dic["orders"]

ecart_approved = (
    (
        pd.to_datetime(orders["order_approved_at"])
        - pd.to_datetime(orders["order_purchase_timestamp"])
    )
    .dropna()
    .mean()
)

ecart_shipped = (
    (
        pd.to_datetime(orders["order_delivered_carrier_date"])
        - pd.to_datetime(orders["order_approved_at"])
    )
    .dropna()
    .mean()
)

ecart_delivered = (
    (
        pd.to_datetime(orders["order_delivered_customer_date"])
        - pd.to_datetime(orders["order_delivered_carrier_date"])
    )
    .dropna()
    .mean()
)

print(ecart_approved)
print(ecart_shipped)
print(ecart_delivered)

0 days 10:25:08.739486
2 days 19:19:15.298850
9 days 07:55:59.254148


### Remplacement des données manquantes par la moyenne